# Recomendador con LLM local (Ollama)

**Introducción al Procesamiento del Lenguaje Natural — TFI 1C 2026**

La segunda estrategia tiene dos pasos. Primero, TF-IDF para preseleccionar películas candidatas, y después le pasamos esa lista al LLM para que elija las 5 finales mirando el historial y la query. La idea es que la parte pesada de filtrar 5000 películas la haga TF-IDF y el LLM solo decida sobre pocas opciones.

### Config y datos

In [9]:
import re
import json

import numpy as np
import pandas as pd

from unidecode import unidecode
from nltk.stem import SnowballStemmer
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

from pydantic import BaseModel
import ollama

In [10]:
MODEL = "qwen2.5:3b"

df = pd.read_csv("data/plots.csv").reset_index(drop=True)
users = pd.read_csv("data/perfiles_usuarios.csv")

### Preselección con TF-IDF

Armamos un texto por película juntando la sinopsis, las keywords y el género. A las keywords y el género los repetimos para que pesen un poco más, porque son más limpios que la sinopsis. Sobre eso entreno el TF-IDF.

In [11]:
sw_es = set(pd.read_csv("data/stop_words_spanish.csv")["word"].astype(str))
STOPWORDS = {unidecode(w).lower() for w in sw_es} | set(ENGLISH_STOP_WORDS)
stemmer = SnowballStemmer("spanish")

def preprocesar(texto):
    t = unidecode(str(texto)).lower()
    t = re.sub(r"[^a-z0-9\s]", " ", t)
    t = re.sub(r"\w*\d\w*", " ", t)
    tokens = [stemmer.stem(w) for w in t.split() if w not in STOPWORDS and len(w) > 2]
    return " ".join(tokens)

df["doc"] = (df["description"].map(preprocesar) + " " +
             (df["keywords"].map(preprocesar) + " ") * 2 +
             (df["genre"].map(preprocesar) + " ") * 2)

vectorizer = TfidfVectorizer(min_df=2, max_df=0.6, ngram_range=(1, 2), sublinear_tf=True)
M = vectorizer.fit_transform(df["doc"])
name_to_idx = {n: i for i, n in enumerate(df["name"])}
hist_cols = [c for c in users.columns if c.startswith("pelicula_")]
M.shape

(4967, 15520)

El shortlist lo armamos combinando dos vectores: las películas del historial y la query, los dos normalizados. Si una película del historial no está en el corpus se salteo. Después ordenamos por similitud coseno y sacamos las que ya vio.

In [12]:
def indices_historial(row):
    idxs = []
    for c in hist_cols:
        titulo = row[c]
        if pd.notna(titulo) and titulo in name_to_idx:
            idxs.append(name_to_idx[titulo])
    return idxs

def shortlist(row, k=15):
    idxs = indices_historial(row)
    v_hist = normalize(np.asarray(M[idxs].mean(axis=0)).reshape(1, -1)) if idxs else None
    vq = vectorizer.transform([preprocesar(row["query"])])
    v_query = normalize(vq).toarray() if vq.nnz else None
    if v_hist is not None and v_query is not None:
        v = 0.5 * v_hist + 0.5 * v_query
    else:
        v = v_hist if v_hist is not None else v_query
    sims = cosine_similarity(v, M).flatten()
    excluir = set(idxs)
    return [i for i in np.argsort(-sims) if i not in excluir][:k]

[df.iloc[i]["name"] for i in shortlist(users.iloc[1])[:8]]

['Gangs of New York',
 'Buenas noches, y buena suerte.',
 '2013: Rescate en L.A.',
 'El expreso de Elmira',
 'La cortina de humo',
 'Maleficio',
 'El asesinato de Richard Nixon',
 'La verdad oculta']

### Prompt

El prompt tiene el rol del sistema y después el contexto del usuario: su historial, lo que pide y la lista de candidatos del shortlist. Le pido que elija 5 nombres exactos de esa lista.

In [13]:
def construir_prompt(row, cands):
    system_prompt = (
        "Sos un sistema de recomendación de películas. A partir del historial de un "
        "usuario y de lo que pide en lenguaje natural, elegís las 5 películas más "
        "adecuadas SOLO de la lista de candidatos. Priorizá la intención de la query "
        "y usá el historial para entender el gusto."
    )

    historial = [row[c] for c in hist_cols if pd.notna(row[c]) and row[c] in name_to_idx]
    candidatos = "\n".join(
        f'- {df.iloc[i]["name"]} [{df.iloc[i]["genre"]}]: {str(df.iloc[i]["description"])[:180]}'
        for i in cands
    )

    user_prompt = (
        f'HISTORIAL: {", ".join(historial)}\n'
        f'QUIERE VER: "{row["query"]}"\n\n'
        f'CANDIDATOS (elegí 5, por nombre exacto):\n{candidatos}'
    )
    return system_prompt, user_prompt

### Llamada al LLM

Usamos un schema de Pydantic para recibir un JSON parseable.

In [14]:
class Recomendacion(BaseModel):
    pelicula: str
    motivo: str

class Recomendaciones(BaseModel):
    recomendaciones: list[Recomendacion]

def recomendar_llm(row, k=12):
    system_prompt, user_prompt = construir_prompt(row, shortlist(row, k))
    response = ollama.chat(
        model=MODEL,
        format=Recomendaciones.model_json_schema(),
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        options={"temperature": 0.2, "num_predict": 400},
        keep_alive="10m",
    )
    return json.loads(response["message"]["content"])

### Resultados

Miramos un perfil definido y uno ambiguo para ver desempeño.

In [15]:
for nombre in ["Rodrigo", "Paula"]:
    row = users[users["nombre"] == nombre].iloc[0]
    print(f'\n{row["nombre"]} ({row["tipo_perfil"]}) — "{row["query"]}"')
    for rec in recomendar_llm(row)["recomendaciones"]:
        print(f'  - {rec["pelicula"]}: {rec["motivo"]}')


Rodrigo (definido) — "Busco algo basado en hechos reales sobre corrupción o poder político"
  - Gangs of New York: Basada en hechos reales, esta película narra una historia de corrupción y poder político durante el famoso período de la Guerra Civil Americana.
  - El expreso de Elmira: Esta biografía muestra la historia inspiradora de un afroamericano que ganó el Trofeo Heisman, lo cual es una excelente representación de hechos reales y corrupción en el deporte.
  - El asesinato de Richard Nixon: Basado en hechos reales, esta película explora la vida y caída del presidente Richard Nixon, incluyendo su papel en la corrupción política.
  - La cara sucia de la ley: Esta película aborda temas como el crimen organizado y la infiltración policial, lo que es relevante para un usuario interesado en hechos reales de poder político y corrupción.
  - El lobo de Wall Street: Basado en la historia real de Jordan Belfort, quien se convirtió en un adinerado agente de bolsa pero luego cayó debido a su

Y ahora el barrido completo sobre los 14 usuarios.

In [16]:
resultados = {}
for _, row in users.iterrows():
    resultados[row["nombre"]] = recomendar_llm(row)

for nombre, res in resultados.items():
    tipo = users[users["nombre"] == nombre]["tipo_perfil"].iloc[0]
    print(f'\n{nombre} ({tipo})')
    for rec in res["recomendaciones"]:
        print(f'  - {rec["pelicula"]}')


Valentina (definido)
  - Tránsito
  - Un San Valentín de muerte
  - Las horas
  - El vigilante nocturno
  - Aflicción

Rodrigo (definido)
  - Gangs of New York
  - El expreso de Elmira
  - 2013: Rescate en L.A.
  - El asesinato de Richard Nixon
  - La cara sucia de la ley

Camila (definido)
  - Joe contra el volcán
  - Niñera a la fuerza
  - Two Much

Tomás (definido)
  - The Giver
  - Fahrenheit 451
  - ¡Olvídate de mí!
  - Matrix Revolutions
  - El único

Lucía (definido)
  - Chicken Run: Evasión en la granja
  - El príncipe de Egipto
  - Sailor Moon Sailor Stars

Martín (definido)
  - Como pez en el agua
  - Atraco en la iglesia
  - Un golpe de altura
  - The Town: Ciudad de ladrones
  - Heat [acción, crimen, drama]

Sofía (definido)
  - Bird [biografía, drama, música]
  - Vidas al límite [biografía, drama, romance]
  - La leyenda del pianista en el océano [drama, música, romance]
  - Acordes y desacuerdos [comedia, drama, música]
  - Once [drama, música, romance]

Diego (definido)